#### Clasificación presencia ausencia

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, StackingClassifier, ExtraTreesClassifier
from sklearn.metrics import classification_report, average_precision_score, precision_recall_curve, confusion_matrix
from sklearn.preprocessing import label_binarize
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import shap


c:\Users\rubar\miniconda3\envs\TFMenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
min_lon, min_lat, max_lon, max_lat =[-61.0, -47.375, -60.0, -44.875]
min_time, max_time = pd.to_datetime("2013-01-01"), pd.to_datetime("2023-12-31")

In [3]:
fishing_ds = xr.open_dataset("../data/processed/targets/cpue_class_0.125.nc")
fishing = fishing_ds["CPUE_class"]
fishing = fishing.fillna(0)

temp_ds = xr.open_dataset("../data/processed/dynamic/to_surface.nc")
temp = temp_ds["to"]
temp = (temp - temp.mean()) / temp.std()
temp = temp.fillna(0)

temp_lag1 = temp.shift(time=1).fillna(0)

temp_bottom_ds = xr.open_dataset("../data/processed/dynamic/temp_bottom.nc")
temp_bottom = temp_bottom_ds["to"]
temp_bottom = (temp_bottom - temp_bottom.mean()) / temp_bottom.std()
temp_bottom = temp_bottom.fillna(0)

temp_bottom_lag1 = temp_bottom.shift(time=1).fillna(0)

chl_ds = xr.open_dataset("../data/processed/dynamic/chl.nc")
chl = chl_ds["CHL"]
chl = (chl - chl.mean()) / chl.std()
chl = chl.fillna(0)

chl_lag1 = chl.shift(time=1).fillna(0)

mixed_ds = xr.open_dataset("../data/processed/dynamic/mixed_layer.nc")
mixed = mixed_ds["mlotst"]
mixed = (mixed - mixed.mean()) / mixed.std()
mixed = mixed.fillna(0)

mixed_lag1 = mixed.shift(time=1).fillna(0)

depth_ds = xr.open_dataset("../data/processed/static/depth.nc")
depth = depth_ds["depth"] 
depth = (depth - depth.mean()) / depth.std()
depth = depth.fillna(0)
depth = depth.broadcast_like(temp)

zo_ds = xr.open_dataset("../data/processed/dynamic/zo_surface.nc")
zo = zo_ds["zo"]
zo = (zo - zo.mean()) / zo.std()
zo = zo.fillna(0)
zo_lag1 = zo.shift(time=1).fillna(0)

so_ds = xr.open_dataset("../data/processed/dynamic/so_surface.nc")
so = so_ds["so"]
so = (so - so.mean()) / so.std()
so = so.fillna(0)

mask_ds = xr.open_dataset("../data/processed/static/fishing_area_mask.nc")
mask = mask_ds["mask"]
mask = mask.broadcast_like(temp)

month = temp["time"].dt.month
month_sin = np.sin(2 * np.pi * month / 12)
month_cos = np.cos(2 * np.pi * month / 12)
month_sin = month_sin.broadcast_like(temp)
month_cos = month_cos.broadcast_like(temp)
month = month.broadcast_like(temp)

year = temp["time"].dt.year
year = (year - year.mean()) / year.std()
year = year.broadcast_like(temp)

lat = (temp["lat"] - temp["lat"].mean()) / temp["lat"].std()
lon = (temp["lon"] - temp["lon"].mean()) / temp["lon"].std()
lat = lat.broadcast_like(temp)
lon = lon.broadcast_like(temp) #se añade como dinámica porque ya se ha corregido la forma

temp, temp_bottom, chl, mixed, fishing, mask, depth, month_sin, month_cos, lat, lon, zo, so, month, year, temp_lag1, mixed_lag1, temp_bottom_lag1, chl_lag1, zo_lag1 = xr.align(temp, temp_bottom, chl, mixed, fishing, mask, depth, month_sin, month_cos, lat, lon, zo, so, month, year, temp_lag1, mixed_lag1, temp_bottom_lag1, chl_lag1, zo_lag1, join="inner")

cropped = lambda da: da.sel(
    lon=slice(min_lon-1, max_lon+1),
    lat=slice(min_lat-2, max_lat+1),
    time=slice(min_time, max_time)
)

temp = cropped(temp)
temp_bottom = cropped(temp_bottom)
chl = cropped(chl)
mixed = cropped(mixed)
mixed_lag1 = cropped(mixed_lag1)
fishing = cropped(fishing)
mask = cropped(mask)
depth = cropped(depth)
zo = cropped(zo)
so = cropped(so)
month = cropped(month)
month_sin = cropped(month_sin)
month_cos = cropped(month_cos)
lat = cropped(lat)
lon = cropped(lon)
year = cropped(year)
temp_lag1 = cropped(temp_lag1)
temp_bottom_lag1 = cropped(temp_bottom_lag1)
chl_lag1 = cropped(chl_lag1)
zo_lag1 = cropped(zo_lag1)


In [4]:
# Target
y = fishing

X = xr.Dataset({
    "temp": temp,
    "temp_bottom": temp_bottom,
    "chl": chl,
    "mixed": mixed,
    "depth": depth,
    "zo": zo,
    "so": so,
    "mask": mask,
    "month": month,
    "lat": lat,
    "lon": lon,
    "year": year,
    "temp_lag1": temp_lag1,
    "temp_bottom_lag1": temp_bottom_lag1,
    "zo_lag1": zo_lag1,


})


time = X.time

train_time = time < np.datetime64("2020-01-01")  
test_time  = ~train_time

X_train_3d = X.sel(time=train_time)
X_test_3d  = X.sel(time=~train_time)

y_train_3d = y.sel(time=train_time)
y_test_3d  = y.sel(time=~train_time)


X_train_df = X_train_3d.to_dataframe().reset_index()
X_test_df  = X_test_3d.to_dataframe().reset_index()

# Convert target
y_train_df = y_train_3d.to_dataframe(name="target").reset_index()
y_test_df  = y_test_3d.to_dataframe(name="target").reset_index()

# Merge target with predictors
train_df = X_train_df.merge(
    y_train_df,
    on=["time", "lat", "lon"]
)

test_df = X_test_df.merge(
    y_test_df,
    on=["time", "lat", "lon"]
)

y_train = train_df["target"].values
y_test = test_df["target"].values

X_train_df = train_df.drop(columns=["target"])
X_test_df = test_df.drop(columns=["target"])


def add_spatial_features(df):
    df = df.copy()

    # cyclical month
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

    # spatial interactions
    df["lat_lon"] = df["lat"] * df["lon"]
    # df["lat2"] = df["lat"] ** 2
    # df["lon2"] = df["lon"] ** 2

    # environmental interactions
    df["temp_depth"] = df["temp"] * df["depth"]
    df["chl_temp"] = df["chl"] * df["temp"]

    return df

X_train_df = add_spatial_features(X_train_df)
X_test_df = add_spatial_features(X_test_df)

X_train_df = X_train_df.drop(columns=["time"])
X_test_df = X_test_df.drop(columns=["time"])


In [5]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_leaf=1,
    min_samples_split=2,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train_df, y_train)

y_pred = rf.predict(X_test_df)
y_prob = rf.predict_proba(X_test_df)

report = classification_report(y_test, y_pred, labels=[1, 2])
print(report)

              precision    recall  f1-score   support

           1       0.53      0.09      0.15      1811
           2       0.36      0.02      0.04      1225

   micro avg       0.49      0.06      0.11      3036
   macro avg       0.44      0.06      0.10      3036
weighted avg       0.46      0.06      0.11      3036



In [6]:
hgb = HistGradientBoostingClassifier(
    max_depth=8,
    learning_rate=0.05,
    max_iter=500,
    class_weight="balanced",
    random_state=42
)

hgb.fit(X_train_df, y_train)

y_pred = hgb.predict(X_test_df)
y_prob = hgb.predict_proba(X_test_df)

report = classification_report(y_test, y_pred, labels=[1, 2])
print(report)




              precision    recall  f1-score   support

           1       0.30      0.71      0.42      1811
           2       0.22      0.31      0.26      1225

   micro avg       0.28      0.55      0.37      3036
   macro avg       0.26      0.51      0.34      3036
weighted avg       0.27      0.55      0.36      3036



In [7]:
lgbm = LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=64,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=42
)

lgbm.fit(X_train_df, y_train)

y_pred = lgbm.predict(X_test_df)
y_prob = lgbm.predict_proba(X_test_df)

report = classification_report(y_test, y_pred, labels=[1, 2])
print(report)



[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002313 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3428
[LightGBM] [Info] Number of data points in the train set: 90720, number of used features: 19
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
              precision    recall  f1-score   support

           1       0.48      0.50      0.49      1811
           2       0.34      0.18      0.24      1225

   micro avg       0.44      0.37      0.41      3036
   macro avg       0.41      0.34      0.36      3036
weighted avg       0.42      0.37      0.39      3036



In [8]:
base_models = [
    ('hgb', HistGradientBoostingClassifier(
        max_depth=6,
        learning_rate=0.05,
        max_iter=500,
        class_weight='balanced',
        random_state=42
    )),
    ('rf', RandomForestClassifier(
        n_estimators=400,
        class_weight='balanced',
        random_state=42
    )),
    ('et', ExtraTreesClassifier(
        n_estimators=400,
        class_weight='balanced',
        random_state=42
    ))
]

stack = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(),
    stack_method='predict_proba',
    cv=5,
    n_jobs=-1
)


stack.fit(X_train_df, y_train)

y_pred = stack.predict(X_test_df)
y_prob = stack.predict_proba(X_test_df)

report = classification_report(y_test, y_pred, labels=[1, 2])
print(report)

              precision    recall  f1-score   support

           1       0.00      0.00      0.00      1811
           2       0.29      0.01      0.02      1225

   micro avg       0.28      0.00      0.01      3036
   macro avg       0.14      0.00      0.01      3036
weighted avg       0.12      0.00      0.01      3036

